# Predictive AI & Modeling Detection

We query the SQL database, and pass the raw features to an Isolation Forest model. This model is an industry-standard unsupervised Machine learning model used extensively for sensor anomaly detection. Additionally, we flag erros, and format an analytical output payload

In [1]:
import sqlite3
import pandas as pd
import json
from sklearn.ensemble import IsolationForest

def run_ai_pipeline():
    # 1. Fetch data using SQL
    conn = sqlite3.connect('factory.db')
    query = "SELECT * FROM Machine_Logs;"
    df = pd.read_sql_query(query, conn)
    conn.close()
    
    if df.empty:
        print("No data found. Please run database_setup.py first!")
        return

    print(f"Extracting {len(df)} telemetry logs from SQL for processing...")

    # 2. Train AI Model for Anomaly Detection
    features = ['voltage', 'current', 'wire_feed_speed', 'temperature']
    
    # contamination=0.02 means we expect roughly 2% of data points to be abnormal anomalies
    model = IsolationForest(contamination=0.02, random_state=42)
    df['anomaly_score'] = model.fit_predict(df[features])
    
    # Map model outputs (-1 = Anomaly, 1 = Normal) to descriptive labels
    df['Status'] = df['anomaly_score'].apply(lambda x: 'Action Required' if x == -1 else 'Healthy')
    
    # 3. Export Cleaned Dataset for Power BI
    df.to_csv('output_insights.csv', index=False)
    print("Analytical dataset exported to 'output_insights.csv' for Power BI dashboards.")
    
    # 4. Filter Critical Issues to trigger Power Automate
    critical_alerts = df[df['Status'] == 'Action Required'].sort_values(by='timestamp', ascending=False)
    
    if not critical_alerts.empty:
        latest_fault = critical_alerts.iloc[0]
        
        # Prepare a lightweight structured JSON payload for Power Automate
        alert_payload = {
            "alert_time": latest_fault['timestamp'],
            "machine": latest_fault['machine_id'],
            "voltage": round(latest_fault['voltage'], 2),
            "current": round(latest_fault['current'], 2),
            "wire_speed": round(latest_fault['wire_feed_speed'], 2),
            "temp": round(latest_fault['temperature'], 2),
            "severity": "CRITICAL" if latest_fault['temperature'] > 90 else "WARNING"
        }
        
        with open('alerts_payload.json', 'w') as f:
            json.dump(alert_payload, f, indent=4)
        print("Latest anomaly payload prepared inside 'alerts_payload.json'.")

if __name__ == "__main__":
    run_ai_pipeline()


Extracting 4320 telemetry logs from SQL for processing...
Analytical dataset exported to 'output_insights.csv' for Power BI dashboards.
Latest anomaly payload prepared inside 'alerts_payload.json'.
